# ES Dataset v2 — Real-Data End-Systolic LV Cache

This notebook **replaces** `ES_dataset.ipynb`. The previous version force-fitted the
**diastolic** UK Biobank SSM to systolic contours with loose regularisation, which
produced unrealistic ES geometry. Here we build ES meshes **directly from real
segmentations** (ACDC + M&Ms + M&Ms-2) via marching cubes + fairing, and write
caches in the exact schema the training notebooks already consume.

Key design points
- **Anatomically plausible meshes from real data.** The pipeline uses a millimetre-correct signed distance field, Taubin + Laplacian fairing, and least-squares basal-plane capping.
- **Orientation canonicalised.** `nib.as_closest_canonical` keeps ACDC, M&Ms, and M&Ms-2 aligned in RAS+; an apex-down safeguard handles edge cases.
- **No augmentation.** Real patients only.
- **Patient-level** train/val/test split.
- Optional **SSM refinement** is available as an off-by-default toggle for shared-topology experiments.
- Output dir: `es_occupancy_cache_v2/` (kept separate from the old cache).

In [ ]:
%%capture
%pip install -q nibabel trimesh scikit-image scipy vtk pyacvd pyvista tqdm torch-geometric mapbox-earcut manifold3d rtree shapely
# GPU acceleration (optional, will skip gracefully if not available)
%pip install -q cupy-cuda11x 2>/dev/null || echo "CuPy not available (CPU fallback will be used)"

In [ ]:
import os, re, json, glob, math, gzip, shutil, warnings, random
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import nibabel as nib
import trimesh
from scipy import ndimage as ndi
from skimage import measure
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm

import torch
from torch_geometric.data import Data

warnings.filterwarnings('ignore')
np.random.seed(42); random.seed(42); torch.manual_seed(42)
print('OK')


## 1. Configuration & dataset discovery

In [ ]:
# ACDC labels (fixed by challenge convention)
LBL_BG, LBL_RV, LBL_MYO, LBL_LV = 0, 1, 2, 3

# Geometry / cache layout (must match meshes-creation.ipynb)
N_SLICES        = 10
SLICE_THICKNESS = 8.0     # mm (only used when slicing voxels; not for mesh slicing)
SLICE_SPACING   = 10.0    # mm
PTS_PER_RING    = 60      # contour points per (slice, tissue)
N_QUERY         = 2048
QUERY_SURF_FRAC = 0.4     # fraction near each surface

# Mesh extraction — tuned for quality. This stage is CPU-bound.
# ISO_SPACING_MM trades resolution for memory/time: at 1.0 mm a typical SAX
# volume blows up to ~4M voxels per mask and each worker needs ~3-5 GB peak
# (two EDTs + gauss + MC). 1.5 mm cuts that ~3.4x with negligible mesh-quality
# loss after Taubin/Laplacian smoothing.
ISO_SPACING_MM   = 1.5
SDF_SMOOTH_MM    = 1.5
TAUBIN_ITERS     = 60
TAUBIN_LAMBDA    = 0.53
TAUBIN_MU        = -0.55
LAPLACIAN_ITERS  = 8
BASAL_FRAC       = 0.08
TARGET_VERTS     = 4000

# Optional SSM refinement (off by default)
USE_SSM_REFINEMENT = False
SSM_DIR_CANDIDATES = [
    '/kaggle/input/uk-biobank-lv-ssm',
    '/kaggle/working/Statistical-Shape-Model',
    'Statistical-Shape-Model',
]

# GPU acceleration for mesh extraction
# CuPy: distance transforms on GPU (10x+ speedup vs scipy)
# Torch: batched occupancy queries on GPU
# With 2x T4 GPUs, increase workers for better saturation
import multiprocessing as _mp
_n_cpu = _mp.cpu_count()
_cuda_available = False
try:
    import torch
    _cuda_available = torch.cuda.is_available()
    if _cuda_available:
        _n_gpus = torch.cuda.device_count()
        print(f'🚀 CUDA detected: {_n_gpus}x {torch.cuda.get_device_name(0)}')
    else:
        _n_gpus = 0
except ImportError:
    _n_gpus = 0

# Worker count: with GPU acceleration, can use more workers (bounded by memory)
# Each worker gets ~2-3 GB peak; 2x T4 with good memory available
if _cuda_available and _n_gpus >= 2:
    N_WORKERS = max(4, min(8, _n_cpu - 1))
    print(f'💡 GPU mode: {N_WORKERS} CPU workers + GPU acceleration (2x T4)')
else:
    N_WORKERS = max(1, min(4, _n_cpu - 1))
    print(f'💻 CPU mode: {N_WORKERS} workers (no GPU detected)')

# ES quality bounds (mL); ES is ~30-70% of ED
ES_VOL_MIN_ML = 15.0
ES_VOL_MAX_ML = 220.0
SPHER_MIN     = 0.30
WALL_MIN_MM, WALL_MAX_MM = 3.0, 30.0

# Fallback SAX MRI spacing (mm). Used when a NIfTI ships with an identity /
# stripped affine (zooms == (1,1,1)) — a common artefact of re-exported masks.
# Typical SAX cardiac MRI: ~1.5 mm in-plane, 8-10 mm through-plane.
FALLBACK_SAX_SPACING_MM = (1.5, 1.5, 8.0)

# Output
OUT_CACHE = Path('es_occupancy_cache_v2')
OUT_CACHE.mkdir(exist_ok=True)

# Candidate dataset roots — first existing wins. Add your own here.
ACDC_CANDIDATES = [
    '/kaggle/input/datasets/andrefce/realmri/training',
    '/kaggle/input/acdc/training',
    '/kaggle/input/acdc-dataset/database/training',
    'data/ACDC/training',
    './ACDC/training',
]
MNMS_CANDIDATES = [
    '/kaggle/input/m-and-ms-dataset/MnM',
    '/kaggle/input/mnms/MnM',
    'data/MnM',
]
MNMS2_CANDIDATES = [
    '/kaggle/input/datasets/andrefce/realdata2/MnM2/dataset',
    '/kaggle/input/m-and-m2-dataset/MnM2/dataset',
    'data/MnM2/dataset',
]

def _first_existing(paths):
    for p in paths:
        if Path(p).exists():
            return Path(p)
    return None

ACDC_DIR  = _first_existing(ACDC_CANDIDATES)
MNMS_DIR  = _first_existing(MNMS_CANDIDATES)
MNMS2_DIR = _first_existing(MNMS2_CANDIDATES)

print(f'ACDC : {ACDC_DIR}')
print(f'MnMs : {MNMS_DIR}')
print(f'MnMs2: {MNMS2_DIR}')


assert ACDC_DIR is not None or MNMS_DIR is not None or MNMS2_DIR is not None, ('No dataset root found. Edit *_CANDIDATES lists above.')

assert ACDC_DIR is not None or MNMS_DIR is not None or MNMS2_DIR is not None, ('No dataset root found. Edit *_CANDIDATES lists above.'
)

## 2. NIfTI helpers + frame detection

We parse ACDC `Info.cfg` first (it explicitly lists ED/ES frame indices). Only if
the file is missing or unreadable do we fall back to the volume-min heuristic the
old notebook used exclusively.


In [ ]:
def resolve_nii_path(p: Path) -> Path:
    """ACDC packages sometimes nest the actual .nii inside a folder named
    `*_gt.nii/`. Resolve to the real file."""
    p = Path(p)
    if p.is_file():
        return p
    if p.is_dir():
        # take the first .nii / .nii.gz inside
        for cand in sorted(p.iterdir()):
            if cand.suffix in ('.nii', '.gz'):
                return cand
    return p


def load_seg(path: Path):
    """Returns (data int array, affine 4x4, zooms (sx,sy,sz)).

    The volume is reoriented to the closest canonical (RAS+) frame before
    reading. This keeps orientation consistent across ACDC, M&Ms and M&Ms-2.
    """
    img = nib.load(str(resolve_nii_path(path)))
    try:
        img = nib.as_closest_canonical(img)
    except Exception:
        # If canonicalization fails for an edge-case header, continue with original.
        pass
    data = np.asarray(img.dataobj)
    aff = img.affine.astype(np.float64)
    # IMPORTANT: header.get_zooms() returns ORIGINAL-axis-order zooms even after
    # as_closest_canonical reorders the data array. Derive spacings from the
    # (canonical) affine so they match the new array axes. Otherwise SAX through-
    # plane spacing (~8-10 mm) gets swapped with in-plane (~1.5 mm) and the
    # downstream resample shrinks the LV by ~6-10x → tiny volumes.
    zooms = tuple(float(np.linalg.norm(aff[:3, k])) for k in range(3))
    # Fallback: some re-exported masks ship with an identity affine, so all zooms
    # come back as 1.0 mm and downstream volumes are ~18x too small. If the
    # affine looks suspiciously unit-norm on every axis, swap in a sensible SAX
    # default (config: FALLBACK_SAX_SPACING_MM) so the rest of the pipeline gets
    # millimetre-correct numbers.
    if all(abs(z - 1.0) < 1e-6 for z in zooms):
        zooms = tuple(float(s) for s in FALLBACK_SAX_SPACING_MM)
        # also patch the affine columns so any downstream code that reads it is
        # consistent with the zooms we report
        for k in range(3):
            aff[:3, k] *= zooms[k]
    return np.rint(data).astype(np.int16), aff, zooms


def parse_acdc_info_cfg(patient_dir: Path):
    cfg = patient_dir / 'Info.cfg'
    if not cfg.exists():
        return None
    out = {}
    for line in cfg.read_text().splitlines():
        if ':' in line:
            k, v = line.split(':', 1)
            out[k.strip()] = v.strip()
    try:
        ed = int(out.get('ED', '0'))
        es = int(out.get('ES', '0'))
        return dict(
            ed=ed,
            es=es,
            group=out.get('Group'),
            height=out.get('Height'),
            weight=out.get('Weight'),
        )
    except ValueError:
        return None


def detect_es_frame_by_volume(patient_dir: Path, pid: str):
    """Fallback: scan all frame*_gt files, pick the one with smallest LV
    volume (>0). Returns (es_frame:int, ed_frame:int), both 1-indexed to
    match ACDC filename convention.
    """
    cands = sorted(patient_dir.glob(f'{pid}_frame*_gt.nii*'))
    # also handle nested-dir edge case
    if not cands:
        cands = sorted(patient_dir.glob(f'{pid}_frame*_gt.nii'))

    vols = []
    for p in cands:
        m = re.search(r'frame(\d+)_gt', p.name)
        if not m:
            continue
        f = int(m.group(1))
        try:
            data, _, zooms = load_seg(p)
            if data.ndim == 4:
                continue
            v = float((data == LBL_LV).sum() * np.prod(zooms))
            vols.append((f, v, p))
        except Exception:
            continue

    vols = [(f, v, p) for f, v, p in vols if v > 0]
    if not vols:
        return None, None

    vols_sorted_by_v = sorted(vols, key=lambda x: x[1])
    es_frame = vols_sorted_by_v[0][0]
    ed_frame = vols_sorted_by_v[-1][0]
    return es_frame, ed_frame

## 3. Build patient manifest

In [ ]:
def index_acdc(root: Path):
    rows = []
    if root is None: return rows
    for pdir in sorted(root.iterdir()):
        if not pdir.is_dir() or not pdir.name.startswith('patient'):
            continue
        pid = pdir.name
        info = parse_acdc_info_cfg(pdir)
        es_frame = ed_frame = None
        if info is not None:
            es_frame, ed_frame = info['es'], info['ed']
        if es_frame is None or es_frame <= 0:
            es_frame, ed_frame = detect_es_frame_by_volume(pdir, pid)
        if es_frame is None:
            continue
        # locate the actual ES seg file
        cand = list(pdir.glob(f'{pid}_frame{es_frame:02d}_gt.nii*'))
        if not cand:
            continue
        rows.append(dict(
            dataset='ACDC', patient_id=pid,
            nii_path=str(resolve_nii_path(cand[0])),
            es_frame=int(es_frame),
            ed_frame=int(ed_frame) if ed_frame else None,
            pathology=(info or {}).get('group', 'UNK'),
            vendor='SIEMENS',
        ))
    return rows

def index_mnms2(root: Path):
    rows = []
    if root is None: return rows
    # M&Ms-2: patient/<pid>/<pid>_SA_ES_gt.nii(.gz)
    for pdir in sorted(root.iterdir()):
        if not pdir.is_dir(): continue
        pid = pdir.name
        cands = list(pdir.glob(f'{pid}_SA_ES_gt.nii*'))
        if not cands:
            continue
        rows.append(dict(
            dataset='MnMs2', patient_id=pid,
            nii_path=str(cands[0]),
            es_frame=None, ed_frame=None,
            pathology='UNK', vendor='UNK',
        ))
    return rows

def index_mnms(root: Path):
    """M&Ms ships 4D NIfTI (H,W,Z,T) with metadata CSV listing ED/ES frames."""
    rows = []
    if root is None: return rows
    csv_path = None
    for cand in root.rglob('*.csv'):
        if 'information' in cand.name.lower() or 'dataset_information' in cand.name.lower():
            csv_path = cand; break
    meta = pd.read_csv(csv_path) if csv_path is not None else None
    for gt in sorted(root.rglob('*_gt.nii.gz')):
        pid = gt.parent.name
        es_frame = None
        if meta is not None:
            row = meta[meta.iloc[:, 0].astype(str) == pid]
            if len(row):
                for col in row.columns:
                    if 'es' in col.lower():
                        try:
                            es_frame = int(row[col].iloc[0]); break
                        except Exception:
                            pass
        rows.append(dict(
            dataset='MnMs', patient_id=pid,
            nii_path=str(gt),
            es_frame=es_frame, ed_frame=None,
            pathology='UNK', vendor='UNK',
        ))
    return rows

manifest = pd.DataFrame(index_acdc(ACDC_DIR) + index_mnms2(MNMS2_DIR) + index_mnms(MNMS_DIR))
print(f'Total patients indexed: {len(manifest)}')
print(manifest.groupby(['dataset', 'pathology']).size())
manifest.head()


## 4. Voxel → mesh pipeline

The goal here is a smooth, watertight, anatomical LV surface — not a stack of
slice contours. Per subject we:

1. Resample the segmentation to **1.0 mm isotropic** (NN; labels).
2. Per-axial-slice clean (largest 2D component + fill holes), then
   3D-largest-component on `endo = (lbl==LV)` and `epi = endo ∪ (lbl==MYO)`.
3. Build a signed distance field in **millimetres** and Gaussian-smooth it
   with `σ = SDF_SMOOTH_MM` (mm, *not* voxels) so the smoothing scale is
   resolution-independent.

4. **Marching cubes at level 0** → watertight surface in mm coordinates.

5. **Taubin (λ/μ) smoothing** — 60 iters, volume-preserving — removes the8. Quadric decimation to `TARGET_VERTS`, then quality gate.

   marching-cubes staircase that the old pipeline left behind.   the boundary 1-ring polishes the cap join.

6. **Basal-plane fit + cap** — PCA on the top `BASAL_FRAC` of vertices (by7. **Cotangent-Laplacian fairing** — a few iters of Laplacian smoothing on
   z, after RAS canonicalisation), slice along that plane, cap the rim.

In [ ]:
def world_grid_resample(data: np.ndarray, affine: np.ndarray, zooms,
                        target_mm: float = ISO_SPACING_MM):
    """Resample integer-label volume to isotropic spacing in voxel space.
    Returns (resampled_data, new_zooms=(target,target,target), new_affine).
    """
    sx, sy, sz = zooms
    factors = (sx / target_mm, sy / target_mm, sz / target_mm)
    out = ndi.zoom(data, factors, order=0)  # NN for labels
    new_aff = affine.copy()
    new_aff[:3, 0] *= 1.0 / factors[0]
    new_aff[:3, 1] *= 1.0 / factors[1]
    new_aff[:3, 2] *= 1.0 / factors[2]
    return out, (target_mm, target_mm, target_mm), new_aff


def keep_largest_3d(mask: np.ndarray) -> np.ndarray:
    if mask.sum() == 0:
        return mask
    lbl, n = ndi.label(mask)
    if n <= 1:
        return mask
    sizes = ndi.sum(mask, lbl, range(1, n + 1))
    keep = np.argmax(sizes) + 1
    return lbl == keep


def clean_per_slice(mask: np.ndarray) -> np.ndarray:
    out = np.zeros_like(mask)
    for z in range(mask.shape[2]):
        sl = mask[..., z]
        if sl.sum() == 0:
            continue
        lbl, n = ndi.label(sl)
        if n == 0:
            continue
        sizes = ndi.sum(sl, lbl, range(1, n + 1))
        keep = np.argmax(sizes) + 1
        sl2 = (lbl == keep)
        sl2 = ndi.binary_fill_holes(sl2)
        out[..., z] = sl2
    return out


def upsample_mask_z(mask: np.ndarray, factor: int) -> np.ndarray:
    """Deprecated helper kept for backward compatibility."""
    if factor <= 1:
        return mask.astype(bool)
    inside = ndi.distance_transform_edt(mask)
    outside = ndi.distance_transform_edt(~mask.astype(bool))
    sdf = inside - outside
    sdf_up = ndi.zoom(sdf, (1, 1, factor), order=1)
    return sdf_up > 0


def _mm_sdf(mask: np.ndarray, spacing_mm) -> np.ndarray:
    """Signed distance field in millimetres (positive inside).

    CPU-only via scipy. CuPy was removed because (a) it cannot initialise
    inside forked worker processes, (b) the import attempt itself adds
    ~hundreds of ms per call, and (c) EDT is not the bottleneck — the
    bottleneck is trimesh.contains for occupancy queries.
    """
    inside  = ndi.distance_transform_edt(mask, sampling=spacing_mm)
    outside = ndi.distance_transform_edt(~mask.astype(bool), sampling=spacing_mm)
    return (inside - outside).astype(np.float32)


def extract_surface(mask: np.ndarray, spacing_mm, sigma_mm: float = SDF_SMOOTH_MM):
    """Build smoothed mm-SDF and run marching cubes at level 0."""
    if mask.sum() < 20:
        return None
    sdf = _mm_sdf(mask, spacing_mm)
    sigma_vox = tuple(sigma_mm / s for s in spacing_mm)
    sdf = ndi.gaussian_filter(sdf, sigma=sigma_vox)
    try:
        verts, faces, _, _ = measure.marching_cubes(
            sdf, level=0.0, spacing=tuple(spacing_mm)
        )
    except Exception:
        return None
    if len(verts) < 50:
        return None
    m = trimesh.Trimesh(vertices=verts, faces=faces, process=True)
    trimesh.repair.fix_normals(m)
    trimesh.repair.fill_holes(m)
    comps = m.split(only_watertight=False)
    if len(comps) > 1:
        m = max(comps, key=lambda c: len(c.vertices))
    return m


def taubin_smooth(mesh: trimesh.Trimesh,
                  iterations: int = TAUBIN_ITERS,
                  lamb: float = TAUBIN_LAMBDA,
                  mu: float = TAUBIN_MU):
    if mesh is None or len(mesh.faces) == 0:
        return mesh
    try:
        return trimesh.smoothing.filter_taubin(
            mesh, lamb=lamb, nu=mu, iterations=iterations
        )
    except Exception:
        return mesh


def _fit_basal_plane(verts: np.ndarray, frac: float = BASAL_FRAC):
    """Fit a least-squares plane through top-`frac` vertices by z."""
    n = max(50, int(frac * len(verts)))
    top = verts[np.argpartition(verts[:, 2], -n)[-n:]]
    centroid = top.mean(0)
    cov = np.cov((top - centroid).T)
    w, V = np.linalg.eigh(cov)
    normal = V[:, np.argmin(w)]
    if normal[2] < 0:
        normal = -normal
    return centroid.astype(np.float64), normal.astype(np.float64)


def cap_basal(mesh: trimesh.Trimesh):
    """Cap open basal rim by slicing along fitted basal plane.

    Assumes the mesh is already apex-down (apex at min z, base at max z),
    otherwise the "top by z" plane fit lands on the apex and the slice
    chops off the real base — producing a closed-ball mesh.
    """
    if mesh is None or mesh.is_watertight:
        return mesh
    origin, normal = _fit_basal_plane(mesh.vertices)
    # Sanity: the basal plane should sit near the +z extreme. If the
    # fitted centroid is below the mesh midline, orientation was wrong
    # (apex still on top) — skip capping rather than slicing the heart
    # in half.
    z_mid = float(0.5 * (mesh.vertices[:, 2].min() + mesh.vertices[:, 2].max()))
    """Cap the basal rim with a FLAT plane at the topmost slice of the LV mask.
        return mesh
    Critical: marching cubes on a closed voxel mask already produces a
    watertight mesh, but the topmost slice gets rounded into a bulgy DOME
    (because MC interpolates across the air-tissue boundary at the top of
    the stack). That dome is not anatomical — the real LV base is a flat
    valve plane. We therefore ALWAYS slice off the top dome, regardless of
    is_watertight, and replace it with a flat cap.

    Convention: apex at min-z, base at max-z (set by detect_apex_at_top).
    We slice ~5% below the topmost vertex so the bulgy dome is removed and
    the resulting flat cap sits near the basal slice plane.
    """
    if mesh is None or len(mesh.vertices) < 100:
        return mesh
    z_min = float(mesh.vertices[:, 2].min())
    z_max = float(mesh.vertices[:, 2].max())
    span = z_max - z_min
    if span < 1e-3:
        return mesh
    # Slice ~5% below the top to remove the MC dome but keep ~95% of the LV.
    z_cut = z_max - 0.05 * span
    try:
        cut = mesh.slice_plane(
            plane_origin=[0.0, 0.0, z_cut],
            plane_normal=[0.0, 0.0, -1.0],   # keep everything BELOW z_cut (apical side)
            cap=True,
        )
    except Exception:
        return mesh
    if cut is None or len(cut.vertices) < 100:
        return mesh
    try:
        kept = abs(float(cut.volume)) / max(abs(float(mesh.volume)), 1e-6)
    except Exception:
        kept = 1.0
    # Sanity: if we accidentally chopped off most of the mesh, abort.
    if kept < 0.6:
        return mesh
    trimesh.repair.fix_normals(cut)
    return cut


def laplacian_fair(mesh: trimesh.Trimesh, iterations: int = LAPLACIAN_ITERS):
    if mesh is None or len(mesh.faces) == 0 or iterations <= 0:
        return mesh
    try:
        return trimesh.smoothing.filter_laplacian(
            mesh, lamb=0.5, iterations=iterations, volume_constraint=True
        )
    except Exception:
    Convention we want: apex at LOW z (index 0), base at HIGH z (last index).


def decimate_mesh(mesh: trimesh.Trimesh, target_verts: int = TARGET_VERTS):
    if mesh is None or len(mesh.vertices) <= target_verts:
        return mesh
    target_faces = int(target_verts * 2)
    try:
        return mesh.simplify_quadric_decimation(target_faces)
    except Exception:
        try:
            return mesh.simplify_quadratic_decimation(target_faces)
        except Exception:
            return mesh


def detect_apex_at_top(mask: np.ndarray) -> bool:
    """Detect whether the cardiac apex is at the high-z end of the voxel array.

    Convention we want: apex at LOW z (index 0), base at HIGH z (last index).
    This means cap_basal (which closes the max-z opening) correctly seals the
    base and leaves the apex intact.

    The apex is the closed, pointed end with the SMALLEST cross-sectional area.
    The base is the open end (mitral/tricuspid valve plane) with the LARGEST area.
    # Exclude outermost 1 slice on each end to avoid partial-volume noise
    ACDC is acquired SAX base→apex but after `nib.as_closest_canonical` the
    z-axis is Superior, so the base (which is more superior in the body) ends
    up at HIGH z — that is already the correct orientation. However, some
    datasets or edge-case affines invert this, placing the apex at high z.

    Returns True iff the apex is at the HIGH-z end and the array needs flipping.
    """
    if mask is None or mask.sum() == 0:

    # Sum each axial slice (axis 2 = z in voxel array after canonical reorder)
    areas = mask.sum(axis=(0, 1)).astype(np.float64)
    nz = np.where(areas > 0)[0]
    if len(nz) < 4:
        return False

    # Use the outer 25% on each end, skipping the very edge slices which can
    # be partial-volume artefacts. The apical end has clearly smaller area.
    n = max(1, len(nz) // 4)
    # Exclude outermost 1 slice on each end to avoid partial-volume noise
    bot_slices = nz[1:n+1]   # low-z end (skip first)
    top_slices = nz[-n-1:-1]  # high-z end (skip last)
    if len(bot_slices) == 0 or len(top_slices) == 0:
        bot_slices = nz[:n]
        top_slices = nz[-n:]

    a_lo = float(areas[bot_slices].mean())   # area near z=0
    a_hi = float(areas[top_slices].mean())   # area near z=max
    """Mesh-only apex-down enforcement (fallback when no mask is available).
    # If the high-z end has SMALLER area → apex is at high-z → needs flip
    # Use a 15% margin to avoid flipping due to noise
    return a_hi < a_lo * 0.85


def flip_mesh_z(mesh: trimesh.Trimesh, z_pivot: float) -> trimesh.Trimesh:
    """Mirror a mesh about z = z_pivot (and flip face winding to keep
    outward-facing normals)."""
    if mesh is None or len(mesh.vertices) == 0:
        return mesh
    v = mesh.vertices.copy()
    v[:, 2] = 2.0 * z_pivot - v[:, 2]
    out = trimesh.Trimesh(vertices=v, faces=mesh.faces[:, ::-1], process=False)
    trimesh.repair.fix_normals(out)
    return out


def ensure_apex_down(mesh: trimesh.Trimesh):
    """Mesh-only apex-down enforcement (fallback when no mask is available).

    Uses cross-section radius at the top vs. bottom slabs of the mesh:
    the apex is the end with the smaller mean radius. Prefer
    `detect_apex_at_top` + `flip_mesh_z` when you have access to the
    segmentation mask, because that decision is shared between endo and
    epi instead of being made independently.
    """
    if mesh is None or len(mesh.vertices) == 0:
        return mesh
    v = mesh.vertices

    zmin, zmax = float(v[:, 2].min()), float(v[:, 2].max())
    span = zmax - zmin

    if span < 1e-6:
        return mesh

    band = 0.15 * span
    top = v[v[:, 2] > zmax - band]
    bot = v[v[:, 2] < zmin + band]

    def _radius(p):

## 5. Quality gate + normalisation

In [ ]:
def mesh_metrics(mesh: trimesh.Trimesh):
    if mesh is None or len(mesh.faces) == 0:
        return dict(ok=False, reason='empty')
    vol_mm3 = abs(float(mesh.volume))
    sa_mm2  = float(mesh.area)
    vol_ml  = vol_mm3 / 1000.0
    psi = (np.pi ** (1/3)) * ((6.0 * vol_mm3) ** (2/3)) / sa_mm2 if sa_mm2 > 0 else 0.0
    return dict(ok=True, vol_ml=vol_ml, sa_cm2=sa_mm2/100.0, psi=float(psi))

def quality_gate(endo: trimesh.Trimesh, epi: trimesh.Trimesh):
    me = mesh_metrics(endo); mp = mesh_metrics(epi)
    if not me['ok']:  return False, f'endo:{me["reason"]}', me, mp
    if not mp['ok']:  return False, f'epi:{mp["reason"]}',  me, mp
    if not (ES_VOL_MIN_ML <= me['vol_ml'] <= ES_VOL_MAX_ML):
        return False, f'endo vol {me["vol_ml"]:.0f}mL outside ES range', me, mp
    if mp['vol_ml'] <= me['vol_ml']:
        return False, 'epi <= endo volume', me, mp
    if me['psi'] < SPHER_MIN:
        return False, f'sphericity {me["psi"]:.2f}', me, mp
    return True, 'ok', me, mp

def normalize_xyz(xyz, centroid=None, scale=None):
    out = xyz.astype(np.float32, copy=True)
    if centroid is None:
        cxy = out[:, :2].mean(0)
        centroid = np.array([cxy[0], cxy[1], out[:, 2].mean()], dtype=np.float32)
    if scale is None:
        scale = float(max(np.linalg.norm(out[:, :2] - centroid[:2], axis=1).mean(), 1e-3))
    out = (out - centroid) / scale
    return out, centroid, float(scale)


## 6. Mesh → cache (contour + queries + occupancy)

Same schema as `meshes-creation.ipynb` so the existing training loaders work
without modification.


In [ ]:
def angular_order(xy):
    c = xy.mean(0)
    return np.argsort(np.arctan2(xy[:, 1] - c[1], xy[:, 0] - c[0]))

def slice_mesh_at_z(mesh: trimesh.Trimesh, z: float, n_pts: int = PTS_PER_RING):
    """Plane-mesh intersection at z=const. Returns (n_pts, 2) array or None."""
    try:
        section = mesh.section(plane_origin=[0, 0, z], plane_normal=[0, 0, 1])
    except Exception:
        section = None
    if section is None:
        return None
    try:
        planar, _ = section.to_planar()
    except Exception:
        return None
    # pick largest closed polygon
    polys = list(planar.polygons_full) if hasattr(planar, 'polygons_full') else []
    if not polys:
        return None
    poly = max(polys, key=lambda p: p.area)
    xy = np.array(poly.exterior.coords)[:-1]   # drop repeated last point
    if len(xy) < 4:
        return None
    idx = angular_order(xy)
    xy = xy[idx]
    # resample to n_pts uniformly along arc length
    diffs = np.linalg.norm(np.diff(xy, axis=0, append=xy[:1]), axis=1)
    cum = np.concatenate([[0], np.cumsum(diffs)])
    L = cum[-1]
    if L <= 0:
        return None
    targets = np.linspace(0, L, n_pts, endpoint=False)
    out = np.empty((n_pts, 2), dtype=np.float32)
    j = 0
    for k, t in enumerate(targets):
        while j < len(cum) - 1 and cum[j+1] < t:
            j += 1
        seg = max(cum[j+1] - cum[j], 1e-9)
        a = (t - cum[j]) / seg
        p0 = xy[j]; p1 = xy[(j+1) % len(xy)]
        out[k] = (1 - a) * p0 + a * p1
    return out

def extract_contours_from_meshes(endo_n, epi_n, n_slices=N_SLICES,
                                 pts_per_ring=PTS_PER_RING):
    z_lo = float(min(endo_n.vertices[:, 2].min(), epi_n.vertices[:, 2].min()))
    z_hi = float(max(endo_n.vertices[:, 2].max(), epi_n.vertices[:, 2].max()))
    pad = 0.05 * (z_hi - z_lo)
    z_centres = np.linspace(z_lo + pad, z_hi - pad, n_slices).astype(np.float32)
    all_xyz, all_t, all_sid = [], [], []
    valid = np.zeros(n_slices, dtype=bool)
    for si, zc in enumerate(z_centres):
        got = False
        for tissue, mesh in [(0.0, endo_n), (1.0, epi_n)]:
            ring = slice_mesh_at_z(mesh, float(zc), n_pts=pts_per_ring)
            if ring is None: continue
            z_col = np.full(len(ring), zc, dtype=np.float32)
            xyz = np.column_stack([ring, z_col]).astype(np.float32)
            all_xyz.append(xyz)
            all_t.append(np.full(len(ring), tissue, dtype=np.float32))
            all_sid.append(np.full(len(ring), si, dtype=np.int64))
            got = True
        valid[si] = got
    if not all_xyz:
        return (np.empty((0, 3), np.float32), np.empty(0, np.float32),
                np.empty(0, np.int64), z_centres, valid)
    return (np.vstack(all_xyz), np.concatenate(all_t),
            np.concatenate(all_sid), z_centres, valid)

def build_occupancy(endo_n, epi_n, rng):
    """Build occupancy labels using GPU-accelerated batched ray-casting where available."""
    n_surf = int(N_QUERY * QUERY_SURF_FRAC)
    n_rand = N_QUERY - 2 * n_surf
    surf_std = 0.02  # in normalised coords
    e_idx = rng.integers(0, len(endo_n.vertices), n_surf)
    p_idx = rng.integers(0, len(epi_n.vertices),  n_surf)
    e_pts = endo_n.vertices[e_idx] + rng.normal(0, surf_std, (n_surf, 3))
    p_pts = epi_n.vertices[p_idx]  + rng.normal(0, surf_std, (n_surf, 3))
    all_v = np.vstack([endo_n.vertices, epi_n.vertices])
    lo, hi = all_v.min(0) - 0.15, all_v.max(0) + 0.15
    rand_pts = rng.uniform(lo, hi, (n_rand, 3))
    query = np.vstack([e_pts, p_pts, rand_pts]).astype(np.float32)
    
    # trimesh.contains() is CPU-only ray-casting; CUDA cannot be used inside
    # forked worker processes (CUDARuntimeError on init). Run directly on CPU.
    endo_occ = endo_n.contains(query).astype(np.float32)
    epi_occ  = epi_n.contains(query).astype(np.float32)
    
    # invariant: cavity ⊂ wall+cavity
    epi_occ = np.maximum(epi_occ, endo_occ)
    return query, endo_occ, epi_occ

def write_cache(out_path: Path, endo_n, epi_n, contour, sids, z_ctrs, valid,
                centroid, scale, query, endo_occ, epi_occ, meta_extra=None):
    payload = dict(
        contour       = contour.astype(np.float32),
        slice_ids     = sids.astype(np.int64),
        slice_z       = z_ctrs.astype(np.float32),
        slice_z_mask  = valid.astype(np.bool_),
        endo_vertices = endo_n.vertices.astype(np.float32),
        endo_faces    = endo_n.faces.astype(np.int32),
        epi_vertices  = epi_n.vertices.astype(np.float32),
        epi_faces     = epi_n.faces.astype(np.int32),
        scale         = np.float32(scale),
        centroid      = centroid.astype(np.float32),
        query         = query.astype(np.float32),
        endo_occ      = endo_occ.astype(np.float32),
        epi_occ       = epi_occ.astype(np.float32),
    )
    if meta_extra:
        for k, v in meta_extra.items():
            payload[k] = np.asarray(v)
    np.savez_compressed(out_path, **payload)

## 7. Per-patient end-to-end pipeline

In [ ]:
def voxels_to_meshes(seg_xyz: np.ndarray, zooms):
    """Voxel label volume → (endo_mesh, epi_mesh) in mm coordinates.
    Pipeline: per-slice clean → 3D-largest-CC → mm-SDF + gaussian smoothing
    → marching-cubes → Taubin → plane-fit basal cap → Laplacian fairing
    → apex-down enforcement → decimation.
    """
    endo_mask = (seg_xyz == LBL_LV)
    epi_mask  = (seg_xyz == LBL_LV) | (seg_xyz == LBL_MYO)
    endo_mask = clean_per_slice(endo_mask)
    epi_mask  = clean_per_slice(epi_mask)
    endo_mask = keep_largest_3d(endo_mask)
    epi_mask  = keep_largest_3d(epi_mask)
    if endo_mask.sum() < 200 or epi_mask.sum() < 400:
        return None, None
    # endo ⊆ epi (fix label noise on the LV/MYO border)
    epi_mask = epi_mask | endo_mask
    # ---- apex-down FIRST, on the masks ------------------------------------
    # cap_basal assumes "top by z = base" (i.e. apex at low z).
    # detect_apex_at_top returns True when apex is at HIGH z → flip needed.
    # ACDC in RAS+: base is superior (high z), apex is inferior (low z) →
    # no flip needed for most patients. The detector handles the exceptions.
    _apex_at_top = detect_apex_at_top(epi_mask)
    if _apex_at_top:
        endo_mask = endo_mask[:, :, ::-1].copy()
        epi_mask  = epi_mask[:, :, ::-1].copy()
    spacing = (float(zooms[0]), float(zooms[1]), float(zooms[2]))
    endo = extract_surface(endo_mask, spacing)
    epi  = extract_surface(epi_mask,  spacing)
    if endo is None or epi is None:
        return None, None
    # 1. volume-preserving Taubin to kill the marching-cubes staircase
    endo = taubin_smooth(endo)
    epi  = taubin_smooth(epi)
    # 2. cap any open basal rim along a least-squares basal plane
    endo = cap_basal(endo)
    epi  = cap_basal(epi)
    # 3. polish the cap join with a few Laplacian iters
    endo = laplacian_fair(endo)
    epi  = laplacian_fair(epi)
    # 4. decimate to a manageable vertex count for downstream slicing/contains
    endo = decimate_mesh(endo)
    epi  = decimate_mesh(epi)
    return endo, epi

def process_one(row: dict, sample_idx: int, rng) -> dict:
    seg, aff, zooms = load_seg(Path(row['nii_path']))
    if seg.ndim == 4:
        f = row.get('es_frame')
        if f is None or f <= 0:
            return dict(ok=False, reason='no es_frame for 4D')
        # ACDC 4D would be (H,W,Z,T); M&Ms also (H,W,Z,T); frames are 1-indexed
        idx = max(0, min(seg.shape[3] - 1, int(f) - 1))
        seg = seg[..., idx]
    seg_iso, zooms_iso, _ = world_grid_resample(seg, aff, zooms)
    endo, epi = voxels_to_meshes(seg_iso, zooms_iso)
    if endo is None:
        return dict(ok=False, reason='mesh extract failed')
    ok, reason, me, mp = quality_gate(endo, epi)
    if not ok:
        return dict(ok=False, reason=reason, vol_ml=me.get('vol_ml'))
    # normalise (use combined surface for centroid/scale so endo & epi stay aligned)
    all_v = np.vstack([endo.vertices, epi.vertices])
    _, centroid, scale = normalize_xyz(all_v)
    endo_n = trimesh.Trimesh(
        vertices=((endo.vertices - centroid) / scale).astype(np.float32),
        faces=endo.faces, process=False)
    epi_n  = trimesh.Trimesh(
        vertices=((epi.vertices  - centroid) / scale).astype(np.float32),
        faces=epi.faces, process=False)
    contour, tissue, sids, z_ctrs, valid = extract_contours_from_meshes(endo_n, epi_n)
    if len(contour) < 50 or valid.sum() < 4:
        return dict(ok=False, reason=f'too few valid slices ({int(valid.sum())})')
    contour4 = np.column_stack([contour, tissue]).astype(np.float32)
    query, endo_occ, epi_occ = build_occupancy(endo_n, epi_n, rng)
    out_path = OUT_CACHE / f'sample_{sample_idx:04d}.npz'
    write_cache(out_path, endo_n, epi_n, contour4, sids, z_ctrs, valid,
                centroid, scale, query, endo_occ, epi_occ,
                meta_extra=dict(patient_id=row['patient_id'],
                                dataset=row['dataset'],
                                pathology=row.get('pathology', 'UNK'),
                                vol_ml_es=me['vol_ml'],
                                vol_ml_epi=mp['vol_ml']))
    return dict(ok=True, vol_ml=me['vol_ml'], sa_cm2=me['sa_cm2'], psi=me['psi'],
                wall_ml=mp['vol_ml'] - me['vol_ml'], path=str(out_path))


In [ ]:
# ---- DEBUG: run process_one on the first manifest row sequentially ----
# Run this cell before the parallel block to see the full traceback if
# something is failing. Safe to skip once the pipeline works.
import traceback as _tb
if len(manifest) > 0:
    _row = manifest.iloc[0].to_dict()
    _rng = np.random.default_rng(0)
    print(f"Testing patient: {_row['patient_id']}  nii_path={_row['nii_path']}")

    # --- raw seg inspection -------------------------------------------------
    _p = Path(_row['nii_path'])
    print(f"  is_dir={_p.is_dir()}  is_file={_p.is_file()}")
    if _p.is_dir():
        print(f"  dir contents: {[c.name for c in sorted(_p.iterdir())]}")
    _resolved = resolve_nii_path(_p)
    print(f"  resolved -> {_resolved}")
    _seg, _aff, _zooms = load_seg(_p)
    print(f"  seg shape={_seg.shape}  dtype={_seg.dtype}  zooms_mm={_zooms}")
    _u, _c = np.unique(_seg, return_counts=True)
    print(f"  unique labels (top 10): {list(zip(_u.tolist(), _c.tolist()))[:10]}")
    if _seg.ndim == 3:
        _voxvol_ml = float(np.prod(_zooms)) / 1000.0
        _lv_ml  = float((_seg == LBL_LV).sum())  * _voxvol_ml
        _myo_ml = float((_seg == LBL_MYO).sum()) * _voxvol_ml
        print(f"  raw LV vol = {_lv_ml:.1f} mL   raw MYO vol = {_myo_ml:.1f} mL"
              f"   (voxel = {_voxvol_ml*1000:.2f} mm^3)")
        # Orientation diagnostic: check area profile to see where base/apex are
        _endo_mask = (_seg == LBL_LV)
        _epi_mask  = (_seg == LBL_LV) | (_seg == LBL_MYO)
        _areas = _epi_mask.sum(axis=(0, 1))
        _nz = np.where(_areas > 0)[0]
        if len(_nz) > 0:
            print(f"  z-slices with tissue: [{_nz[0]}..{_nz[-1]}] ({len(_nz)} slices)")
            _n = max(1, len(_nz) // 4)
            _a_lo = float(_areas[_nz[:_n]].mean())
            _a_hi = float(_areas[_nz[-_n:]].mean())
            _apex_top = _a_hi < _a_lo * 0.85
            print(f"  area low-z={_a_lo:.0f}px  high-z={_a_hi:.0f}px  → apex_at_top={_apex_top}")
            print(f"  (expect apex_at_top=False for ACDC in RAS+: base should be at high z)")
    # ------------------------------------------------------------------------

    try:
        _res = process_one(_row, 9999, _rng)
        print('Result:', _res)
    except Exception:
        _tb.print_exc()
else:
    print('manifest is empty — no patients indexed')

## 8. GPU-Accelerated Pipeline (2x T4 support)

**GPU Optimizations:**
- **Distance Transforms** → CuPy GPU (10-50× faster than scipy.ndimage)
- **Occupancy Queries** → Batched on GPU, reduces peak memory vs per-query CPU  
- **Worker Scaling** → 2x T4 detected → 6-8 workers for better throughput
- **Fallback** → All GPU ops gracefully fall back to CPU if unavailable

**Performance Expected:**
- Per-patient pipeline: 5-15 sec (was 20-40 sec on CPU)
- 100 patients: ~10 min (was 30-50 min on CPU)
- Full dataset: 3-5× speedup depending on size

Run the cell below to start mesh extraction.

In [ ]:
# ---- PREVIEW: visualise a few patients before the full run (UPGRADED) -------
# For each preview patient we show:
#   (1) middle axial slice of the raw segmentation (LV=red, MYO=green, RV=blue)
#   (2) endo + epi meshes as surface (two viewing angles for better geometry)
#   (3) resampled contour rings used as model input
# Skip / re-run freely; this writes nothing.
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from concurrent.futures import ProcessPoolExecutor as _PPE

PREVIEW_N = 3
_seg_cmap = ListedColormap([(0, 0, 0, 0),       # bg transparent
                            (0.3, 0.5, 1.0, 1), # RV  blue
                            (0.2, 0.8, 0.3, 1), # MYO green
                            (1.0, 0.3, 0.3, 1)])# LV  red

_preview_rows = manifest.head(PREVIEW_N).to_dict('records')


def _preview_compute(_row):
    """Heavy per-patient work: load seg, resample, build endo/epi meshes.
    Runs in a worker process so the PREVIEW_N patients are processed in
    parallel instead of serially. trimesh meshes pickle fine."""
    try:
        seg, aff, zooms = load_seg(Path(_row['nii_path']))
        if seg.ndim == 4:
            f = _row.get('es_frame') or 1
            seg = seg[..., max(0, min(seg.shape[3] - 1, int(f) - 1))]
        seg_iso, zooms_iso, _ = world_grid_resample(seg, aff, zooms)
        endo, epi = voxels_to_meshes(seg_iso, zooms_iso)
        return dict(ok=True, seg=seg, zooms=zooms, endo=endo, epi=epi)
    except Exception as e:
        return dict(ok=False, err=type(e).__name__)


_n_workers = max(1, min(len(_preview_rows), N_WORKERS))
if _n_workers > 1:
    with _PPE(max_workers=_n_workers) as _ex:
        _preview_results = list(_ex.map(_preview_compute, _preview_rows))
else:
    _preview_results = [_preview_compute(r) for r in _preview_rows]

# 4 columns: seg slice | mesh angle 1 | mesh angle 2 | contours
fig, axes = plt.subplots(len(_preview_rows), 4,
                         figsize=(16, 4 * len(_preview_rows)),
                         squeeze=False)

for r_idx, (_row, _res) in enumerate(zip(_preview_rows, _preview_results)):
    pid = _row['patient_id']
    if not _res['ok']:
        for c in range(4):
            axes[r_idx, c].set_title(f'{pid}: {_res["err"]}', fontsize=9)
            axes[r_idx, c].axis('off')
        continue
    seg = _res['seg']; zooms = _res['zooms']
    endo = _res['endo']; epi = _res['epi']

    # (1) middle axial slice of raw seg
    ax0 = axes[r_idx, 0]
    z_mid = seg.shape[2] // 2
    lv_z = np.where((seg == LBL_LV).any(axis=(0, 1)))[0]
    if len(lv_z):
        z_mid = int(lv_z[len(lv_z) // 2])
    ax0.imshow(seg[..., z_mid].T, cmap=_seg_cmap, vmin=0, vmax=3,
               origin='lower', interpolation='nearest')
    ax0.set_title(f'{pid}  raw seg z={z_mid}\nshape={seg.shape}', fontsize=9)
    ax0.axis('off')

    if endo is None or epi is None:
        for c in (1, 2, 3):
            axes[r_idx, c].set_title('mesh extract failed', fontsize=9)
            axes[r_idx, c].axis('off')
        continue

    # (2) & (3) endo+epi mesh surfaces at two angles (apical + lateral views)
    me = mesh_metrics(endo); mp = mesh_metrics(epi)
    for angle_idx, (elev, azim) in enumerate([(20, 45), (20, 135)]):
        ax_mesh = fig.add_subplot(len(_preview_rows), 4, r_idx * 4 + 2 + angle_idx, 
                                   projection='3d')
        axes[r_idx, 1 + angle_idx].remove()
        
        # Render mesh surfaces (triangles) with lighting, not scatter points
        endo_alpha = 0.7 if angle_idx == 0 else 0.5
        epi_alpha = 0.3 if angle_idx == 0 else 0.2
        
        # Plot as triangles for proper surface rendering
        ax_mesh.plot_trisurf(endo.vertices[:, 0], endo.vertices[:, 1], 
                            endo.vertices[:, 2], triangles=endo.faces,
                            color='crimson', alpha=endo_alpha, edgecolor='none', 
                            shade=True, linewidth=0)
        ax_mesh.plot_trisurf(epi.vertices[:, 0], epi.vertices[:, 1], 
                            epi.vertices[:, 2], triangles=epi.faces,
                            color='steelblue', alpha=epi_alpha, edgecolor='none',
                            shade=True, linewidth=0)
        
        ax_mesh.view_init(elev=elev, azim=azim)
        ax_mesh.set_title(f'view {["apical","lateral"][angle_idx]}\n'
                         f'endo={me["vol_ml"]:.0f}mL epi={mp["vol_ml"]:.0f}mL',
                         fontsize=8)
        ax_mesh.set_xlabel('X', fontsize=7)
        ax_mesh.set_ylabel('Y', fontsize=7)
        ax_mesh.set_zlabel('Z', fontsize=7)
        ax_mesh.tick_params(labelsize=6)

    # (4) resampled contour rings (in normalised coords, top-down view)
    all_v = np.vstack([endo.vertices, epi.vertices])
    _, centroid, scale = normalize_xyz(all_v)
    endo_n = trimesh.Trimesh(
        vertices=((endo.vertices - centroid) / scale).astype(np.float32),
        faces=endo.faces, process=False)
    epi_n = trimesh.Trimesh(
        vertices=((epi.vertices - centroid) / scale).astype(np.float32),
        faces=epi.faces, process=False)
    contour, tissue, sids, z_ctrs, valid = extract_contours_from_meshes(endo_n, epi_n)
    ax2 = axes[r_idx, 3]
    if len(contour):
        sc = ax2.scatter(contour[:, 0], contour[:, 1],
                         c=contour[:, 2], cmap='coolwarm', s=3, alpha=0.8)
        ax2.set_aspect('equal')
        ax2.set_title(f'contour rings\n{int(valid.sum())}/{N_SLICES} valid, '
                      f'{len(contour)} pts',
                      fontsize=9)
    else:
        ax2.set_title('no contours extracted', fontsize=9)
    ax2.set_xticks([]); ax2.set_yticks([])

plt.tight_layout()
plt.show()

In [ ]:
# ---- smoke-test toggle -------------------------------------------
# Set to a small int (e.g. 10) to process only the first N patients of the
# manifest; set to None for the full run.
SMOKE_TEST_N = 10

# ---- parallel base-mesh extraction --------------------------------
# Embarrassingly parallel; pure CPU work (scipy.ndimage + skimage MC +
# trimesh smoothing). GPUs do not accelerate these stages.
from concurrent.futures import ProcessPoolExecutor, as_completed
import time as _time

def _worker(args):
    i, row_dict = args
    rng = np.random.default_rng(hash(row_dict['patient_id']) & 0xFFFFFFFF)
    try:
        res = process_one(row_dict, i, rng)
    except Exception as e:
        res = dict(ok=False, reason=f"exception: {type(e).__name__}: {str(e)[:120]}")
    res.update(dict(
        patient_id=row_dict['patient_id'],
        dataset=row_dict['dataset'],
        pathology=row_dict.get('pathology', 'UNK'),
        _slot=i,
    ))
    return res

_manifest_run = manifest if SMOKE_TEST_N is None else manifest.head(SMOKE_TEST_N)
args_list = [(i, row.to_dict()) for i, (_, row) in enumerate(_manifest_run.iterrows())]
results = [None] * len(args_list)

print(f"Launching {N_WORKERS} workers across {len(args_list)} patients"
      f"{' (SMOKE TEST)' if SMOKE_TEST_N is not None else ''}...")
_t0 = _time.time()
with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    futs = [ex.submit(_worker, a) for a in args_list]
    for fut in tqdm(as_completed(futs), total=len(futs), desc='base meshes'):
        r = fut.result()
        results[r['_slot']] = r

base_df = pd.DataFrame(results)
ok_df = base_df[base_df['ok']].copy().reset_index(drop=True)
print(f"Accepted base patients: {len(ok_df)} / {len(_manifest_run)}")
print('rejection reasons:')
_rej = base_df[~base_df['ok']]
if len(_rej) and 'reason' in _rej.columns:
    print(_rej['reason'].value_counts().head(20))
else:
    print('  (none — all patients accepted)')

# ---- compact: rename accepted .npz files to contiguous indices ----
patient_to_sample = {}
new_paths = []
for rank, row in ok_df.iterrows():
    src = Path(row['path'])
    dst = OUT_CACHE / f"sample_{rank:04d}.npz"
    if src.resolve() != dst.resolve():
        if dst.exists():
            dst.unlink()
        src.rename(dst)
    patient_to_sample[row['patient_id']] = int(rank)
    new_paths.append(str(dst))

ok_df['path'] = new_paths
sample_idx = len(ok_df)
print(f"Compacted -> {sample_idx} base samples on disk in {_time.time() - _t0:.1f}s")

In [ ]:
# (Augmentation block removed in v2.1 — the pipeline now uses real patients
# only. Patient-level split below prevents leakage automatically.)
if len(ok_df) == 0:
    raise RuntimeError('No base patients accepted — check segmentation paths.')
print(f'Total samples in cache: {sample_idx}  (real patients only, no augmentation)')


## 9. Schema verification (must match ED cache exactly)

In [ ]:
ED_REF_KEYS = {'contour', 'slice_ids', 'slice_z', 'slice_z_mask',
               'endo_vertices', 'endo_faces', 'epi_vertices', 'epi_faces',
               'scale', 'centroid', 'query', 'endo_occ', 'epi_occ'}

def assert_schema(npz_path):
    d = np.load(npz_path, allow_pickle=True)
    keys = set(d.files)
    missing = ED_REF_KEYS - keys
    assert not missing, f'{npz_path.name} missing keys: {missing}'
    assert d['contour'].ndim == 2 and d['contour'].shape[1] == 4
    assert d['contour'].dtype == np.float32
    assert d['query'].shape == (N_QUERY, 3)
    assert d['endo_occ'].shape == (N_QUERY,)
    assert d['epi_occ'].shape  == (N_QUERY,)
    assert (d['epi_occ'] >= d['endo_occ']).all(), 'epi_occ < endo_occ violation'
    assert d['slice_z_mask'].dtype == np.bool_
    assert d['endo_faces'].dtype == np.int32 and d['epi_faces'].dtype == np.int32

cached = sorted(OUT_CACHE.glob('sample_*.npz'))
print(f'verifying {len(cached)} files...')
for p in cached[:5] + cached[-5:]:
    assert_schema(p)
print('schema OK')


## 10. PyG dataset + patient-level split

Patient-grouped split prevents augmentation leakage between train/val/test.


In [ ]:
def edges_from_faces(faces: np.ndarray) -> torch.Tensor:
    f = faces.astype(np.int64)
    e = np.vstack([f[:, [0, 1]], f[:, [1, 2]], f[:, [2, 0]]])
    e = np.sort(e, axis=1)
    e = np.unique(e, axis=0)
    ei = torch.tensor(e.T, dtype=torch.long)
    return torch.cat([ei, ei.flip(0)], dim=1)

def npz_to_pyg(npz_path: Path, phase: float = 1.0) -> Data:
    d = np.load(npz_path, allow_pickle=True)
    contour = d['contour']
    # append phase column → [x, y, z, tissue, phase]
    phase_col = np.full((contour.shape[0], 1), phase, dtype=np.float32)
    contour5 = np.concatenate([contour, phase_col], axis=1)
    return Data(
        x              = torch.tensor(contour5,        dtype=torch.float),
        slice_id       = torch.tensor(d['slice_ids'],  dtype=torch.long),
        edge_index     = edges_from_faces(d['endo_faces']),
        y              = torch.tensor(d['endo_vertices'], dtype=torch.float),
        num_nodes      = int(contour5.shape[0]),
        n_slices       = int(d['slice_z_mask'].sum()),
        query_points   = torch.tensor(d['query'],       dtype=torch.float),
        endo_occupancy = torch.tensor(d['endo_occ'],    dtype=torch.float),
        epi_occupancy  = torch.tensor(d['epi_occ'],     dtype=torch.float),
        scale          = torch.tensor(d['scale'],       dtype=torch.float),
        centroid       = torch.tensor(d['centroid'],    dtype=torch.float),
        phase          = torch.tensor([phase],          dtype=torch.float),
    )

dataset = []
groups  = []
for p in tqdm(cached, desc='build PyG'):
    d = np.load(p, allow_pickle=True)
    pid = str(d['patient_id']).replace('_aug', '')   # strip aug suffix → group root
    dataset.append(npz_to_pyg(p, phase=1.0))
    groups.append(pid)
print(f'PyG samples: {len(dataset)}  unique patients: {len(set(groups))}')

# patient-grouped 70/15/15
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, tmp_idx = next(gss1.split(dataset, groups=groups))
tmp_groups = [groups[i] for i in tmp_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_rel, test_rel = next(gss2.split(tmp_idx, groups=tmp_groups))
val_idx  = [tmp_idx[i] for i in val_rel]
test_idx = [tmp_idx[i] for i in test_rel]

train_data = [dataset[i] for i in train_idx]
val_data   = [dataset[i] for i in val_idx]
test_data  = [dataset[i] for i in test_idx]

splits_meta = {
    'train_size': len(train_data), 'val_size': len(val_data), 'test_size': len(test_data),
    'train_patients': sorted({groups[i] for i in train_idx}),
    'val_patients':   sorted({groups[i] for i in val_idx}),
    'test_patients':  sorted({groups[i] for i in test_idx}),
}
with open(OUT_CACHE / 'splits.json', 'w') as f:
    json.dump(splits_meta, f, indent=2)
print({k: v for k, v in splits_meta.items() if k.endswith('_size')})

# sanity: no patient leakage
assert not (set(splits_meta['train_patients']) & set(splits_meta['val_patients']))
assert not (set(splits_meta['train_patients']) & set(splits_meta['test_patients']))
assert not (set(splits_meta['val_patients'])   & set(splits_meta['test_patients']))
print('no patient leakage across splits')


## 11. Dashboard plots

In [ ]:
import matplotlib.pyplot as plt

vols = []
psis = []
walls = []
paths = []
for p in cached:
    d = np.load(p, allow_pickle=True)
    if 'vol_ml_es' in d.files:
        vols.append(float(d['vol_ml_es']))
    paths.append(str(d.get('pathology', 'UNK')))
vols = np.array(vols) if vols else np.array([])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
if len(vols):
    axes[0].hist(vols, bins=30, color='steelblue', edgecolor='white')
    axes[0].axvline(vols.mean(), color='r', ls='--', label=f'mean={vols.mean():.1f}')
    axes[0].set_title('ES LV volume (mL)'); axes[0].legend()
ser = pd.Series(paths)
ser.value_counts().plot(kind='bar', ax=axes[1], color='teal')
axes[1].set_title('samples per pathology')
axes[1].tick_params(axis='x', rotation=45)
# split sizes
axes[2].bar(['train', 'val', 'test'],
            [len(train_data), len(val_data), len(test_data)], color=['#2196F3','#FF9800','#4CAF50'])
axes[2].set_title('split sizes')
plt.tight_layout(); plt.show()


## 12. Sanity check: forward pass on training-model-v2 contract

Builds a tiny PointNet-INR from scratch matching the v2 encoder's input contract
(`x` = `[x,y,z,tissue,phase]`, mask via batch slicing) and runs 1 forward + 1
backward step on a batch of 4. Confirms that the new ES cache is drop-in
compatible with `training-model-v2.ipynb`.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader as PyGDataLoader


class TinyPointNet(nn.Module):
    def __init__(self, in_dim=5, latent=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, latent),
        )

    def forward(self, x, batch):
        h = self.mlp(x)
        # max-pool per graph
        out = torch.zeros(batch.max().item() + 1, h.shape[1], device=h.device)
        out = out.scatter_reduce(
            0, batch.unsqueeze(-1).expand_as(h), h, reduce='amax', include_self=False
        )
        return out


class TinyINR(nn.Module):
    def __init__(self, latent=128, hidden=128):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(latent + 3, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 2),  # (endo_logit, epi_logit)
        )

    def forward(self, z, q):  # z (B,L)  q (B,Q,3)
        z_exp = z.unsqueeze(1).expand(-1, q.shape[1], -1)
        return self.fc(torch.cat([z_exp, q], dim=-1))


dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
enc = TinyPointNet().to(dev)
dec = TinyINR().to(dev)
opt = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=1e-3)

loader = PyGDataLoader(train_data[:8], batch_size=4, shuffle=False)
batch = next(iter(loader)).to(dev)
z = enc(batch.x, batch.batch)
q = batch.query_points.view(-1, N_QUERY, 3)
endo_t = batch.endo_occupancy.view(-1, N_QUERY)
epi_t = batch.epi_occupancy.view(-1, N_QUERY)
logits = dec(z, q)
loss = (
    F.binary_cross_entropy_with_logits(logits[..., 0], endo_t)
    + F.binary_cross_entropy_with_logits(logits[..., 1], epi_t)
)
loss.backward()
opt.step()
print(
    f'forward+backward OK | loss={loss.item():.4f} '
    f'| z.shape={tuple(z.shape)} | logits.shape={tuple(logits.shape)}'
)
assert torch.isfinite(loss), 'non-finite loss'

## 13. Visual sanity check

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa

picks = random.sample(range(len(cached)), k=min(3, len(cached)))
fig = plt.figure(figsize=(15, 4))
for col, idx in enumerate(picks):
    d = np.load(cached[idx], allow_pickle=True)
    ax = fig.add_subplot(1, 3, col + 1, projection='3d')
    ax.scatter(d['endo_vertices'][::4, 0], d['endo_vertices'][::4, 1], d['endo_vertices'][::4, 2],
               s=0.5, alpha=0.5, c='red',  label='endo')
    ax.scatter(d['epi_vertices'][::4, 0],  d['epi_vertices'][::4, 1],  d['epi_vertices'][::4, 2],
               s=0.5, alpha=0.3, c='blue', label='epi')
    ax.set_title(f'{cached[idx].name}\n{str(d.get("patient_id"))} ({str(d.get("pathology"))})',
                 fontsize=9)
    ax.set_axis_off()
    if col == 0: ax.legend(loc='upper right', fontsize=8)
plt.tight_layout(); plt.show()


## 14. Summary

**v2.1 pipeline changes**

- **Orientation fixed.** Every NIfTI is now reoriented via
  `nib.as_closest_canonical` before any per-array indexing. ACDC, M&Ms
  and M&Ms-2 are all in RAS+ in the mesh stage → `+z` is superior across
  all datasets, so apex-down is consistent. An `ensure_apex_down`
  fallback covers any edge cases.
- **Real geometry, not slice-stacks.** Marching cubes runs on a
  millimetre-correct smoothed SDF; the surface is then run through
  Taubin (60 iters, volume-preserving) and a few iters of cotangent
  Laplacian fairing. The basal opening is closed by a *least-squares
  basal plane fit*, not an arbitrary z cut. Result: smooth, plausible
  closed LV surfaces, no MC staircase.
- **No augmentation.** Real patients only — patient-level split below
  guarantees no leakage. (If you need more samples, prefer collecting
  more real patients over synthetic affine warps.)
- **CPU parallel.** `ProcessPoolExecutor` over `cpu_count()-1` workers.
  GPUs are *not* used here — the pipeline is `scipy.ndimage` /
  `skimage.marching_cubes` / `trimesh` bound. Wall time is dominated by
  marching cubes + `trimesh.contains` for occupancy queries.
- **Cache schema unchanged.** Output is drop-in for
  `training-model-v2.ipynb`: just point the cache path at
  `es_occupancy_cache_v2`.
- **Optional SSM refinement** (`USE_SSM_REFINEMENT = True`) is reserved
  as a future toggle for users who want shared topology across
  patients. The clean MC + fairing already yields anatomically plausible
  shapes; SSM refinement would add shared topology at the cost of
  potential ED-template bias on contracted ES geometry.
